# 03. Baseline and Hybrid Models

**Цель:** последовательно проверить модели от простых к более сложным.

Логика переходов:
1. Сначала Popularity, потому что EDA показал сильный popularity bias.
2. Затем Content-based, потому что есть описания, жанры, актёры, ключевые слова.
3. Затем ALS, чтобы проверить collaborative filtering.
4. Затем Hybrid, потому что отдельно popularity/content/ALS имеют ограничения.
5. Затем Cold-Aware Hybrid, потому что есть cold items.
6. Затем Demographic/Adaptive, потому что есть full cold и semi-cold пользователи.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

K = 10
EVAL_N_USERS = 5000

items_path =  "/content/drive/MyDrive/items_processed.csv"
users_path =  "/content/drive/MyDrive/users_processed.csv"
interactions_path =   "/content/drive/MyDrive/interactions_processed.csv"

print("items:", items_path)
print("users:", users_path)
print("interactions:", interactions_path)

items = pd.read_csv(items_path)
users = pd.read_csv(users_path)
interactions = pd.read_csv(interactions_path, parse_dates=["last_watch_dt"])

print("items:", items.shape)
print("users:", users.shape)
print("interactions:", interactions.shape)

display(items.head())
display(users.head())
display(interactions.head())

items: /content/drive/MyDrive/items_processed.csv
users: /content/drive/MyDrive/users_processed.csv
interactions: /content/drive/MyDrive/interactions_processed.csv
items: (15963, 14)
users: (441316, 5)
interactions: (5476251, 5)


,item_id,content_type,title,title_orig,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords,release_year_cat
0,10711,film,поговори с ней,Hable con ella,"драмы, зарубежные, детективы, мелодрамы",испания,False,16.0,unknown,педро альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ...",2000-2010
1,2508,film,голые перцы,Search Party,"зарубежные, приключения, комедии",сша,False,16.0,unknown,скот армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео...",2010-2020
2,10716,film,тактическая сила,Tactical Force,"криминал, зарубежные, триллеры, боевики, комедии",канада,False,16.0,unknown,адам п. калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг...",2010-2020
3,7868,film,45 лет,45 Years,"драмы, зарубежные, мелодрамы",великобритания,False,16.0,unknown,эндрю хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю...",2010-2020
4,16268,film,все решает мгновение,NaN,"драмы, спорт, советские, мелодрамы",ссср,False,12.0,ленфильм,виктор садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж...",1970-1980


,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,M,True
1,962099,age_18_24,income_20_40,M,False
2,1047345,age_45_54,income_40_60,F,False
3,721985,age_45_54,income_20_40,F,False
4,704055,age_35_44,income_60_90,F,False


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72
1,699317,1659,2021-05-29,8317,100
2,656683,7107,2021-05-09,10,0
3,864613,7638,2021-07-05,14483,100
4,964868,9506,2021-04-30,6725,100


In [ ]:
if "watched_pct" not in interactions.columns:
    interactions["watched_pct"] = 0

if "total_dur" not in interactions.columns:
    interactions["total_dur"] = 0

interactions["watched_pct"] = interactions["watched_pct"].fillna(0).astype(float)
interactions["total_dur"] = interactions["total_dur"].fillna(0).astype(float)

if "interaction_score" not in interactions.columns:
    max_dur = interactions["total_dur"].max()

    if max_dur > 0:
        interactions["interaction_score"] = (
            0.7 * interactions["watched_pct"] / 100
            + 0.3 * np.log1p(interactions["total_dur"]) / np.log1p(max_dur)
        )
    else:
        interactions["interaction_score"] = interactions["watched_pct"] / 100

interactions["interaction_score"] = interactions["interaction_score"].fillna(0)

display(interactions[[
    "user_id",
    "item_id",
    "last_watch_dt",
    "watched_pct",
    "total_dur",
    "interaction_score"
]].head())

,user_id,item_id,last_watch_dt,watched_pct,total_dur,interaction_score
0,176549,9506,2021-05-11,72.0,4250.0,0.641698
1,699317,1659,2021-05-29,100.0,8317.0,0.848761
2,656683,7107,2021-05-09,0.0,10.0,0.039520
3,864613,7638,2021-07-05,100.0,14483.0,0.857902
4,964868,9506,2021-04-30,100.0,6725.0,0.845260


## Метрики

In [ ]:
import numpy as np
import pandas as pd

def precision_at_k(recs, true_items, k=10):
    recs = recs[:k]
    if len(recs) == 0:
        return 0
    return len(set(recs) & set(true_items)) / k

def recall_at_k(recs, true_items, k=10):
    recs = recs[:k]
    if len(true_items) == 0:
        return 0
    return len(set(recs) & set(true_items)) / len(set(true_items))

def evaluate_model(recommend_func, test_df, users_sample, k=10):
    test_grouped = test_df.groupby("user_id")["item_id"].apply(list).to_dict()
    precisions, recalls = [], []

    for user_id in users_sample:
        if user_id not in test_grouped:
            continue

        true_items = test_grouped[user_id]
        recs = recommend_func(user_id, k=k)

        precisions.append(precision_at_k(recs, true_items, k))
        recalls.append(recall_at_k(recs, true_items, k))

    return {
        f"precision@{k}": np.mean(precisions) if len(precisions) else np.nan,
        f"recall@{k}": np.mean(recalls) if len(recalls) else np.nan,
        "n_users": len(precisions)
    }

def get_eval_sample(users_list, n=5000, seed=42, replace=False):
    rng = np.random.default_rng(seed)
    users_list = np.array(list(users_list))
    if len(users_list) == 0:
        return []
    if len(users_list) <= n and not replace:
        return list(users_list)
    size = n if replace else min(n, len(users_list))
    return list(rng.choice(users_list, size=size, replace=replace))

In [ ]:
split_date = interactions["last_watch_dt"].quantile(0.8)

train = interactions[
    interactions["last_watch_dt"] <= split_date
].copy()

test = interactions[
    interactions["last_watch_dt"] > split_date
].copy()

print("Split date:", split_date)

print()
print("TRAIN")
print("Interactions:", len(train))
print("Users:", train["user_id"].nunique())
print("Items:", train["item_id"].nunique())

print()
print("TEST")
print("Interactions:", len(test))
print("Users:", test["user_id"].nunique())
print("Items:", test["item_id"].nunique())

Split date: 2021-08-04 00:00:00

TRAIN
Interactions: 4423600
Users: 819309
Items: 15331

TEST
Interactions: 1052651
Users: 312327
Items: 8863


In [ ]:
train_items = set(train["item_id"].unique())
test_items = set(test["item_id"].unique())

cold_items = list(test_items - train_items)

print("Cold items:", len(cold_items))
print(
    "Cold item share:",
    round(
        len(cold_items) / test["item_id"].nunique() * 100,
        2
    ),
    "%"
)

Cold items: 375
Cold item share: 4.23 %


In [ ]:
train_users = set(train["user_id"].unique())
test_users = set(test["user_id"].unique())

full_cold_users = list(test_users - train_users)

user_hist = (
    train
    .groupby("user_id")
    .size()
    .reset_index(name="n_interactions")
)

test_user_set = set(test["user_id"].unique())

very_cold_users = user_hist[
    user_hist["n_interactions"] <= 2
]["user_id"].tolist()

semi_cold_users = user_hist[
    (user_hist["n_interactions"] >= 3)
    & (user_hist["n_interactions"] <= 5)
]["user_id"].tolist()

medium_users = user_hist[
    (user_hist["n_interactions"] >= 6)
    & (user_hist["n_interactions"] <= 9)
]["user_id"].tolist()

warm_users = user_hist[
    user_hist["n_interactions"] >= 10
]["user_id"].tolist()

# оставляем только пользователей из test

very_cold_users = [
    u for u in very_cold_users
    if u in test_user_set
]

semi_cold_users = [
    u for u in semi_cold_users
    if u in test_user_set
]

medium_users = [
    u for u in medium_users
    if u in test_user_set
]

warm_users = [
    u for u in warm_users
    if u in test_user_set
]

segments = {
    "full_cold_0": full_cold_users,
    "very_cold_1_2": very_cold_users,
    "semi_cold_3_5": semi_cold_users,
    "medium_6_9": medium_users,
    "warm_10_plus": warm_users,
}

segment_stats = pd.DataFrame({
    "segment": list(segments.keys()),
    "n_users": [len(v) for v in segments.values()]
})

display(segment_stats)

,segment,n_users
0,full_cold_0,142870
1,very_cold_1_2,46344
2,semi_cold_3_5,38243
3,medium_6_9,28614
4,warm_10_plus,56256


## 1. Popularity baseline

**Гипотеза:** если в данных сильный popularity bias, то простая рекомендация популярных объектов будет сильным baseline.

**Зачем:** все сложные модели должны сравниваться с этим уровнем.

In [ ]:
popular_items = train.groupby("item_id").size().sort_values(ascending=False).index.tolist()
user_train_items = train.groupby("user_id")["item_id"].apply(set).to_dict()

def recommend_popularity(user_id, k=10):
    seen = user_train_items.get(user_id, set())
    recs = []
    for item_id in popular_items:
        if item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break
    return recs

## 2. TF-IDF Content-Based

**Гипотеза:** контентные признаки помогут персонализировать рекомендации и работать с cold items.

**Почему переходим сюда:** в `items` есть описания, жанры, актёры, режиссёры, ключевые слова.

In [ ]:
if "item_text" not in items.columns:
    text_cols = ["title", "genres", "countries", "studios", "directors", "actors", "description", "keywords", "release_year_cat", "content_type"]
    for col in text_cols:
        if col not in items.columns:
            items[col] = ""
        items[col] = items[col].fillna("").astype(str)
    items["item_text"] = items[text_cols].agg(" ".join, axis=1)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
item_tfidf = tfidf.fit_transform(items["item_text"].fillna("").astype(str))

item_id_to_idx = dict(zip(items["item_id"], range(len(items))))
idx_to_item_id = dict(zip(range(len(items)), items["item_id"]))

def build_user_profile(user_id):
    hist = train[train["user_id"] == user_id]
    if hist.empty:
        return None

    idxs, weights = [], []
    for _, row in hist.iterrows():
        item_id = row["item_id"]
        if item_id in item_id_to_idx:
            idxs.append(item_id_to_idx[item_id])
            weights.append(row["interaction_score"])

    if len(idxs) == 0:
        return None

    weights = np.array(weights)
    profile = item_tfidf[idxs].multiply(weights.reshape(-1, 1)).sum(axis=0)
    profile = normalize(np.asarray(profile))
    return profile

def recommend_tfidf_content(user_id, k=10):
    profile = build_user_profile(user_id)
    if profile is None:
        return recommend_popularity(user_id, k)

    seen = user_train_items.get(user_id, set())
    scores = cosine_similarity(profile, item_tfidf).ravel()
    ranked_idx = np.argsort(-scores)

    recs = []
    for idx in ranked_idx:
        item_id = idx_to_item_id[idx]
        if item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break
    return recs

## 3. ALS Collaborative Filtering

**Гипотеза:** ALS выявит скрытые предпочтения пользователей через user-item матрицу.

**Ожидаемое ограничение:** высокая разреженность и короткие истории могут снижать качество ALS.

In [ ]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 72.6 MB/s eta 0:00:00


In [ ]:
try:
    import implicit

    user_ids = train["user_id"].unique()
    item_ids = items["item_id"].unique()

    user_to_idx = {u: i for i, u in enumerate(user_ids)}
    item_to_idx_als = {i: j for j, i in enumerate(item_ids)}
    idx_to_item_als = {j: i for i, j in item_to_idx_als.items()}

    train_als = train[
        train["user_id"].isin(user_to_idx) &
        train["item_id"].isin(item_to_idx_als)
    ].copy()

    rows = train_als["user_id"].map(user_to_idx)
    cols = train_als["item_id"].map(item_to_idx_als)
    values = train_als["interaction_score"].astype(float)

    user_item_matrix = csr_matrix(
        (values, (rows, cols)),
        shape=(len(user_to_idx), len(item_to_idx_als))
    )

    als_model = implicit.als.AlternatingLeastSquares(
        factors=64,
        regularization=0.05,
        iterations=20,
        random_state=RANDOM_STATE
    )

    als_model.fit(user_item_matrix)

    ALS_AVAILABLE = True
    print("ALS trained")

except Exception as e:
    ALS_AVAILABLE = False
    print("ALS skipped:", e)

def recommend_als(user_id, k=10):
    if not ALS_AVAILABLE:
        return recommend_popularity(user_id, k)
    if user_id not in user_to_idx:
        return recommend_popularity(user_id, k)

    user_idx = user_to_idx[user_id]
    recs, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=k,
        filter_already_liked_items=True
    )
    return [idx_to_item_als[i] for i in recs]

  0%|          | 0/20 [00:00<?, ?it/s]

ALS trained


## 4. Hybrid Content + Popularity

**Гипотеза:** popularity даёт сильный общий сигнал, content добавляет персонализацию. Их комбинация должна быть лучше каждой компоненты по отдельности.

In [ ]:
item_popularity = train.groupby("item_id").size()
item_popularity = np.log1p(item_popularity)

pop_score = pd.Series(0, index=items["item_id"])
pop_score.loc[item_popularity.index] = item_popularity
if pop_score.max() > 0:
    pop_score = pop_score / pop_score.max()

item_pop_score_array = items["item_id"].map(pop_score).fillna(0).values
cold_item_set = set(cold_items)

def recommend_hybrid_content_popularity(user_id, k=10, alpha=0.7):
    profile = build_user_profile(user_id)
    if profile is None:
        return recommend_popularity(user_id, k)

    seen = user_train_items.get(user_id, set())
    content_scores = cosine_similarity(profile, item_tfidf).ravel()
    scores = alpha * content_scores + (1 - alpha) * item_pop_score_array
    ranked_idx = np.argsort(-scores)

    recs = []
    for idx in ranked_idx:
        item_id = idx_to_item_id[idx]
        if item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break
    return recs

## 5. Cold-Aware Hybrid

**Гипотеза:** для cold items нет popularity/collaborative сигнала, поэтому для них нужно увеличить вес content-компоненты.

In [ ]:
def recommend_cold_aware_hybrid(user_id, k=10, alpha_warm=0.6, alpha_cold_item=0.95):
    profile = build_user_profile(user_id)
    if profile is None:
        return recommend_popularity(user_id, k)

    seen = user_train_items.get(user_id, set())
    content_scores = cosine_similarity(profile, item_tfidf).ravel()

    alpha_array = np.full(len(items), alpha_warm)
    for item_id in cold_item_set:
        if item_id in item_id_to_idx:
            alpha_array[item_id_to_idx[item_id]] = alpha_cold_item

    scores = alpha_array * content_scores + (1 - alpha_array) * item_pop_score_array
    ranked_idx = np.argsort(-scores)

    recs = []
    for idx in ranked_idx:
        item_id = idx_to_item_id[idx]
        if item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break
    return recs

## 6. Demographic и Adaptive Strategy

**Гипотеза:** для full cold users истории нет, поэтому нужны user features. Для разных сегментов cold-start логично использовать разные стратегии.

In [ ]:
users_demo = users.copy()
for col in ["age", "sex", "income", "kids_flg"]:
    if col not in users_demo.columns:
        users_demo[col] = "unknown"
    users_demo[col] = users_demo[col].fillna("unknown").astype(str)

users_demo["demo_group"] = (
    users_demo["age"].astype(str) + "_" +
    users_demo["sex"].astype(str) + "_" +
    users_demo["income"].astype(str) + "_" +
    users_demo["kids_flg"].astype(str)
)

train_demo = train.merge(users_demo[["user_id", "demo_group"]], on="user_id", how="left")
train_demo["demo_group"] = train_demo["demo_group"].fillna("unknown")

demo_popularity = (
    train_demo
    .groupby(["demo_group", "item_id"])
    .size()
    .reset_index(name="cnt")
    .sort_values(["demo_group", "cnt"], ascending=[True, False])
)

demo_recs = demo_popularity.groupby("demo_group")["item_id"].apply(list).to_dict()
user_demo_group = dict(zip(users_demo["user_id"], users_demo["demo_group"]))

def recommend_demographic(user_id, k=10):
    seen = user_train_items.get(user_id, set())
    group = user_demo_group.get(user_id, "unknown")
    candidates = demo_recs.get(group, popular_items)

    recs = []
    for item_id in candidates:
        if item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break

    if len(recs) < k:
        for item_id in popular_items:
            if item_id not in seen and item_id not in recs:
                recs.append(item_id)
            if len(recs) >= k:
                break

    return recs

def get_user_history_len(user_id):
    return len(user_train_items.get(user_id, set()))

def recommend_adaptive_strategy(user_id, k=10):
    n = get_user_history_len(user_id)

    if n == 0:
        return recommend_demographic(user_id, k)
    elif n <= 2:
        return recommend_tfidf_content(user_id, k)
    elif n <= 5:
        return recommend_hybrid_content_popularity(user_id, k, alpha=0.8)
    elif n <= 9:
        return recommend_hybrid_content_popularity(user_id, k, alpha=0.65)
    else:
        return recommend_hybrid_content_popularity(user_id, k, alpha=0.5)

## 7. Итоговая оценка

In [ ]:
models = {
    "Popularity": recommend_popularity,
    "TF-IDF content": recommend_tfidf_content,
    "ALS": recommend_als,
    "Hybrid content + popularity": recommend_hybrid_content_popularity,
    "Cold-aware hybrid": recommend_cold_aware_hybrid,
    "Demographic": recommend_demographic,
    "Adaptive strategy": recommend_adaptive_strategy,
}

overall_users = get_eval_sample(test["user_id"].unique(), n=EVAL_N_USERS)

overall_results = []
for name, func in models.items():
    print("Evaluating:", name)
    res = evaluate_model(func, test, overall_users, k=K)
    res["model"] = name
    res["scenario"] = "overall"
    overall_results.append(res)

overall_results_df = pd.DataFrame(overall_results)
overall_results_df = overall_results_df[["scenario", "model", f"precision@{K}", f"recall@{K}", "n_users"]]
display(overall_results_df.sort_values(f"precision@{K}", ascending=False))

Evaluating: Popularity
Evaluating: TF-IDF content
Evaluating: ALS
Evaluating: Hybrid content + popularity
Evaluating: Cold-aware hybrid
Evaluating: Demographic
Evaluating: Adaptive strategy


,scenario,model,precision@10,recall@10,n_users
4,overall,Cold-aware hybrid,0.05822,0.261264,5000
3,overall,Hybrid content + popularity,0.05666,0.255935,5000
5,overall,Demographic,0.05584,0.255192,5000
0,overall,Popularity,0.05524,0.252738,5000
6,overall,Adaptive strategy,0.05216,0.234735,5000
1,overall,TF-IDF content,0.04270,0.201941,5000
2,overall,ALS,0.04256,0.195657,5000


In [ ]:
segment_results = []

for segment_name, segment_users in segments.items():
    eval_users = get_eval_sample(segment_users, n=EVAL_N_USERS)
    print("\\nSegment:", segment_name, "n_eval_users:", len(eval_users))

    for model_name, func in models.items():
        print("   ", model_name)
        res = evaluate_model(func, test, eval_users, k=K)
        res["model"] = model_name
        res["scenario"] = segment_name
        segment_results.append(res)

segment_results_df = pd.DataFrame(segment_results)
segment_results_df = segment_results_df[["scenario", "model", f"precision@{K}", f"recall@{K}", "n_users"]]
display(segment_results_df.sort_values(["scenario", f"precision@{K}"], ascending=[True, False]))

precision_pivot = segment_results_df.pivot(index="model", columns="scenario", values=f"precision@{K}")
recall_pivot = segment_results_df.pivot(index="model", columns="scenario", values=f"recall@{K}")

display(precision_pivot)
display(recall_pivot)

\nSegment: full_cold_0 n_eval_users: 5000
    Popularity
    TF-IDF content
    ALS
    Hybrid content + popularity
    Cold-aware hybrid
    Demographic
    Adaptive strategy
\nSegment: very_cold_1_2 n_eval_users: 5000
    Popularity
    TF-IDF content
    ALS
    Hybrid content + popularity
    Cold-aware hybrid
    Demographic
    Adaptive strategy
\nSegment: semi_cold_3_5 n_eval_users: 5000
    Popularity
    TF-IDF content
    ALS
    Hybrid content + popularity
    Cold-aware hybrid
    Demographic
    Adaptive strategy
\nSegment: medium_6_9 n_eval_users: 5000
    Popularity
    TF-IDF content
    ALS
    Hybrid content + popularity
    Cold-aware hybrid
    Demographic
    Adaptive strategy
\nSegment: warm_10_plus n_eval_users: 5000
    Popularity
    TF-IDF content
    ALS
    Hybrid content + popularity
    Cold-aware hybrid
    Demographic
    Adaptive strategy


,scenario,model,precision@10,recall@10,n_users
5,full_cold_0,Demographic,0.06630,0.340522,5000
6,full_cold_0,Adaptive strategy,0.06630,0.340522,5000
0,full_cold_0,Popularity,0.06622,0.340754,5000
1,full_cold_0,TF-IDF content,0.06622,0.340754,5000
2,full_cold_0,ALS,0.06622,0.340754,5000
3,full_cold_0,Hybrid content + popularity,0.06622,0.340754,5000
4,full_cold_0,Cold-aware hybrid,0.06622,0.340754,5000
25,medium_6_9,Cold-aware hybrid,0.04628,0.180312,5000
27,medium_6_9,Adaptive strategy,0.04566,0.177300,5000
24,medium_6_9,Hybrid content + popularity,0.04392,0.171609,5000


scenario,full_cold_0,medium_6_9,semi_cold_3_5,very_cold_1_2,warm_10_plus
model,,,,,
ALS,0.06622,0.01970,0.01798,0.01322,0.02370
Adaptive strategy,0.06630,0.04566,0.03992,0.01640,0.04354
Cold-aware hybrid,0.06622,0.04628,0.05040,0.05182,0.04350
Demographic,0.06630,0.04334,0.04758,0.05108,0.03800
Hybrid content + popularity,0.06622,0.04392,0.04820,0.04880,0.04176
Popularity,0.06622,0.04112,0.04686,0.05082,0.03690
TF-IDF content,0.06622,0.01880,0.01872,0.01640,0.01792


scenario,full_cold_0,medium_6_9,semi_cold_3_5,very_cold_1_2,warm_10_plus
model,,,,,
ALS,0.340754,0.067822,0.072944,0.058403,0.052408
Adaptive strategy,0.340522,0.177300,0.173864,0.086092,0.116223
Cold-aware hybrid,0.340754,0.180312,0.218892,0.247109,0.114784
Demographic,0.340522,0.169943,0.204106,0.240937,0.103887
Hybrid content + popularity,0.340754,0.171609,0.209176,0.235525,0.109495
Popularity,0.340754,0.159049,0.200370,0.240432,0.098895
TF-IDF content,0.340754,0.076798,0.083666,0.086092,0.047022


## Интерпретация результатов

Полученные результаты согласуются с выводами EDA и экспериментального дизайна. Данные имеют высокую разреженность и выраженный popularity bias, поэтому простая модель Popularity уже показывает достаточно сильное качество.

Наилучшие overall-результаты показали гибридные модели. **Cold-Aware Hybrid** оказался лучшим по Precision@10, а **Hybrid Content + Popularity** занял второе место. Это означает, что объединение информации о популярности объектов и их контентных характеристик работает лучше, чем использование каждого источника по отдельности.

Чистый TF-IDF content-based подход оказался слабее popularity baseline. Это показывает, что похожесть фильмов по описаниям, жанрам и ключевым словам не всегда совпадает с фактическим пользовательским выбором.

ALS также не смог превзойти popularity baseline. Это важный результат: collaborative filtering требует достаточно плотной истории взаимодействий, а в данном датасете много пользователей с короткой историей и высокая разреженность user-item матрицы.

Сегментный анализ показывает, что для пользователей с короткой и средней историей особенно хорошо работают гибридные подходы. Для full cold users различия между моделями минимальны, так как у этих пользователей нет истории взаимодействий в train.

## Итоговый вывод по базовым моделям

В рамках исследования были рассмотрены popularity-based, content-based, collaborative и гибридные рекомендательные модели.

По результатам overall-оценки лучшей моделью среди базовых подходов стал **Cold-Aware Hybrid** со значениями **Precision@10 = 0.05822** и **Recall@10 = 0.26126**. Это подтверждает гипотезу о том, что в условиях холодного старта наиболее эффективными оказываются гибридные подходы, сочетающие сигнал популярности и контентные признаки объектов.

Модель **Hybrid Content + Popularity** также показала сильный результат (**Precision@10 = 0.05666**, **Recall@10 = 0.25594**), что подтверждает полезность объединения контентного и popularity-сигналов.

Модель **Popularity** достигла **Precision@10 = 0.05524** и **Recall@10 = 0.25274**. Это достаточно высокий результат для простого baseline и он согласуется с выводами EDA о наличии выраженного popularity bias в данных.

Content-based модель на основе **TF-IDF** показала более низкое качество (**Precision@10 = 0.04270**, **Recall@10 = 0.20194**). Это означает, что использование только текстовой и метаинформации об объектах недостаточно для точного прогнозирования будущих просмотров пользователей.

Collaborative filtering на основе **ALS** также показал низкое качество относительно popularity baseline (**Precision@10 = 0.04256**, **Recall@10 = 0.19566**). Это может быть связано с высокой разреженностью user-item матрицы и большим количеством пользователей с короткой историей взаимодействий, что было выявлено на этапе EDA.

Дополнительный анализ по cold-start сегментам показал, что **Cold-Aware Hybrid** стабильно входит в число лучших моделей почти во всех сегментах пользователей: very cold, semi-cold, medium и warm. Для full cold users большинство моделей дают близкие результаты, поскольку для таких пользователей отсутствует история взаимодействий в train, и модели вынуждены использовать fallback-стратегии.

Таким образом, результаты подтверждают основную гипотезу исследования: при наличии выраженной проблемы холодного старта и высокой разреженности данных наиболее устойчивыми являются гибридные рекомендательные подходы.